# Climate Data – A hands-on python course
Author: Pedro Herrera Lormendez (pedrolormendez@gmail.com)

**Updated 2025:** Enhanced with ensemble statistics, model evaluation metrics, uncertainty quantification, and IPCC AR6 findings

## Future Climate: Global Climate Models (GCMs) and CMIP6

### What are Global Climate Models?

**Global Climate Models (GCMs)** are complex computer simulations that represent Earth's climate system by solving mathematical equations for:
* **Atmosphere:** Temperature, pressure, winds, humidity, clouds
* **Oceans:** Currents, temperature, salinity, sea ice
* **Land surface:** Vegetation, soil moisture, snow cover, rivers
* **Ice sheets:** Glaciers, ice caps, sea ice dynamics
* **Biogeochemistry:** Carbon cycle, vegetation feedback

### CMIP6 (Coupled Model Intercomparison Project Phase 6)

**CMIP6** is a collaborative framework coordinating climate model experiments from ~100 modeling centers worldwide.

**Key features:**
* Standardized experiments for model comparison
* Multiple emission scenarios (SSPs - Shared Socioeconomic Pathways)
* Improved resolution and physics compared to CMIP5
* Central to IPCC AR6 (2021-2023) assessment

**Reference:** Eyring et al. (2016), GMD; IPCC AR6 WG1 (2021)

### SSP Scenarios (Shared Socioeconomic Pathways)

SSPs combine **socioeconomic narratives** with **radiative forcing levels**:

| Scenario | Description | End-of-century warming (2081-2100 vs 1850-1900) | Policy context |
|----------|-------------|----------------------------------------|----------------|
| **SSP1-1.9** | Very low emissions | ~1.4°C (1.0-1.8°C) | Well below 2°C target |
| **SSP1-2.6** | Low emissions | ~1.8°C (1.3-2.4°C) | Paris Agreement compatible |
| **SSP2-4.5** | Intermediate | ~2.7°C (2.1-3.5°C) | Middle-of-the-road |
| **SSP3-7.0** | High emissions | ~3.6°C (2.8-4.6°C) | Regional rivalry |
| **SSP5-8.5** | Very high emissions | ~4.4°C (3.3-5.7°C) | Fossil-fueled development |

**Numbers refer to radiative forcing in W/m² by 2100**

**Key differences from RCPs (CMIP5):**
* SSPs include socioeconomic narratives (population, GDP, technology)
* More internally consistent storylines
* Better representation of air quality policies

**IPCC AR6 findings:**
* Human influence has warmed the climate at an unprecedented rate (~1.1°C since 1850-1900)
* Recent warming rate: 0.2°C per decade (2006-2015 vs 1850-1900)
* Every increment of warming matters
* 1.5°C will be reached in early 2030s under all scenarios
* Limiting warming to 1.5°C requires immediate, rapid, and sustained reductions

**Reference:** IPCC AR6 WG1 (2021), Summary for Policymakers

### Available CMIP6 data sources

**1. Climate Data Store (CDS):**
* [CMIP6 Climate Projections](https://cds.climate.copernicus.eu/cdsapp#!/dataset/projections-cmip6?tab=overview)
* Pre-processed, quality-controlled
* Easier to use for beginners

**2. ESGF (Earth System Grid Federation):**
* [ESGF Portal](https://esgf-node.llnl.gov/)
* Original CMIP6 data archive
* More models and variables
* Requires more processing

**3. IPCC WG1 Interactive Atlas:**
* [IPCC Interactive Atlas](https://interactive-atlas.ipcc.ch/)
* Regional aggregations
* Multi-model means
* User-friendly interface

**4. Climate Extreme Indices:**
* [CMIP6 Extreme Indices](https://cds.climate.copernicus.eu/cdsapp#!/dataset/sis-extreme-indices-cmip6?tab=form)
* Heat stress indicators
* Precipitation extremes

### Data for this notebook

We use **MPI-ESM1-2-LR** (Max Planck Institute Earth System Model, Low Resolution):

**To download from CDS:**
1. Navigate to CMIP6 Climate Projections
2. Select:
   - Temporal resolution: **Monthly**
   - Experiments: **Historical, SSP1-2.6, SSP2-4.5, SSP3-7.0, SSP5-8.5**
   - Variable: **Near-surface air temperature (tas)**
   - Model: **MPI-ESM1-2-LR**
   - Years: Historical (1950-2014), Projections (2015-2100)
   - Region: **Whole available region**

**Or download directly:**
* [Historical 1950-2014](https://drive.google.com/file/d/1-UTWOSxfqtxMYN1Z4G0t2oNBAjmMXNn-/view?usp=sharing)
* [SSP1-2.6 2015-2100](https://drive.google.com/file/d/1-bgt6R_JsdLOzCJDG8sRejGiD1hlq2oS/view?usp=sharing)
* [SSP2-4.5 2015-2100](https://drive.google.com/file/d/1-XkTT4IuZ2dRflVw2kt0bwQXpdf_q6xe/view?usp=sharing)
* [SSP3-7.0 2015-2100](https://drive.google.com/file/d/1-QA4aj2ah4kxcZTrRpdHj2LZcefiAnfm/view?usp=sharing)
* [SSP5-8.5 2015-2100](https://drive.google.com/file/d/1-PX1JsdZ5yOODLBlkIJnND_GoH1c3Zj8/view?usp=sharing)

**For multi-model ensembles:** You would download data from multiple models (e.g., CESM2, UKESM1-0-LL, MIROC6, etc.)

### Importing necessary modules

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import warnings

# Configure plotting
plt.rcParams['figure.dpi'] = 100
warnings.filterwarnings('ignore', category=RuntimeWarning)

### Loading CMIP6 data

In [ ]:
# Define file paths
file_hist = '../data/cmip6/tas_Amon_MPI-ESM1-2-LR_historical_r1i1p1f1_gn_19500116-20141216_v20190710.nc'
file_ssp1 = '../data/cmip6/tas_Amon_MPI-ESM1-2-LR_ssp126_r1i1p1f1_gn_20150116-20991216_v20190710.nc'
file_ssp2 = '../data/cmip6/tas_Amon_MPI-ESM1-2-LR_ssp245_r1i1p1f1_gn_20150116-20991216_v20190710.nc'
file_ssp3 = '../data/cmip6/tas_Amon_MPI-ESM1-2-LR_ssp370_r1i1p1f1_gn_20150116-20991216_v20190710.nc'
file_ssp5 = '../data/cmip6/tas_Amon_MPI-ESM1-2-LR_ssp585_r1i1p1f1_gn_20150116-20991216_v20190710.nc'

# Load datasets
try:
    ds_hist = xr.open_dataset(file_hist)
    ds_ssp1 = xr.open_dataset(file_ssp1)
    ds_ssp2 = xr.open_dataset(file_ssp2)
    ds_ssp3 = xr.open_dataset(file_ssp3)
    ds_ssp5 = xr.open_dataset(file_ssp5)
    print("✓ All datasets loaded successfully")
    
    # Display model metadata
    print(f"\nModel: {ds_hist.attrs.get('source_id', 'N/A')}")
    print(f"Institution: {ds_hist.attrs.get('institution', 'N/A')}")
    print(f"Variant: {ds_hist.attrs.get('variant_label', 'N/A')}")
    print(f"Grid: {ds_hist.attrs.get('grid_label', 'N/A')}")
    
except FileNotFoundError as e:
    print(f"Error loading files: {e}")
    print("Please download CMIP6 data from the links above.")
    raise

In [ ]:
# Extract temperature variable and convert to °C
tas_hist = ds_hist.tas - 273.15
tas_ssp1 = ds_ssp1.tas - 273.15
tas_ssp2 = ds_ssp2.tas - 273.15
tas_ssp3 = ds_ssp3.tas - 273.15
tas_ssp5 = ds_ssp5.tas - 273.15

# Add units attributes
for tas in [tas_hist, tas_ssp1, tas_ssp2, tas_ssp3, tas_ssp5]:
    tas.attrs['units'] = '°C'

print(f"Historical period: {pd.to_datetime(tas_hist.time[0].values).year} - {pd.to_datetime(tas_hist.time[-1].values).year}")
print(f"Projection period: {pd.to_datetime(tas_ssp1.time[0].values).year} - {pd.to_datetime(tas_ssp1.time[-1].values).year}")
print(f"\nSpatial resolution: {len(tas_hist.lat)} × {len(tas_hist.lon)} grid points")

### Initial visualization: Global mean temperature

In [ ]:
# Compute global mean and yearly average
tas_hist_global = tas_hist.mean(dim=('lat', 'lon')).groupby('time.year').mean()
tas_ssp1_global = tas_ssp1.mean(dim=('lat', 'lon')).groupby('time.year').mean()
tas_ssp2_global = tas_ssp2.mean(dim=('lat', 'lon')).groupby('time.year').mean()
tas_ssp3_global = tas_ssp3.mean(dim=('lat', 'lon')).groupby('time.year').mean()
tas_ssp5_global = tas_ssp5.mean(dim=('lat', 'lon')).groupby('time.year').mean()

# Create plot
plt.figure(figsize=(15, 7))

# Plot each scenario
plt.plot(tas_hist_global.year, tas_hist_global.values, 
         color='black', linewidth=2, label='Historical', zorder=5)
plt.plot(tas_ssp1_global.year, tas_ssp1_global.values, 
         color='darkgreen', linewidth=2, label='SSP1-2.6 (Low emissions)')
plt.plot(tas_ssp2_global.year, tas_ssp2_global.values, 
         color='blue', linewidth=2, label='SSP2-4.5 (Intermediate)')
plt.plot(tas_ssp3_global.year, tas_ssp3_global.values, 
         color='orange', linewidth=2, label='SSP3-7.0 (High emissions)')
plt.plot(tas_ssp5_global.year, tas_ssp5_global.values, 
         color='red', linewidth=2, label='SSP5-8.5 (Very high emissions)')

plt.axvline(x=2015, color='gray', linestyle='--', linewidth=1, alpha=0.7, label='Projection start')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Global Mean Temperature (°C)', fontsize=12)
plt.title('CMIP6 Historical and Projected Global Mean Temperature\nMPI-ESM1-2-LR Model', 
          fontsize=13, fontweight='bold')
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('cmip6_global_temperature_scenarios.png', dpi=300, bbox_inches='tight')
plt.show()

### ⚠️ Important: Model Limitations and Biases

When using raw CMIP6 data, be aware of:

**1. Model Biases:**
* Systematic errors in representing climate processes
* Vary by region, season, and variable
* Can be 2-5°C for regional temperatures

**2. Representation Limitations:**
* Models simplify complex processes
* Parameterizations for sub-grid phenomena
* Limited resolution (typically 100-200 km)

**3. Regional Specificity:**
* Global models better for large-scale patterns
* Regional models needed for local impacts
* Biases more pronounced in some regions (Arctic, tropics, mountains)

**4. Baseline Calibration:**
* Use **anomalies** rather than absolute values
* Bias correction needed for impact studies
* Compare with observations (e.g., ERA5)

**5. Inter-Model Variability:**
* Different models give different projections
* Ensemble mean more reliable than single model
* Spread represents uncertainty

**Best practice:** Use **anomalies relative to a reference period** rather than absolute values.

**Reference:** Maraun (2016), BAMS; Eyring et al. (2021), Nature Climate Change

---
## Model Evaluation: Comparing with Observations

Before trusting future projections, we evaluate the model's performance in reproducing historical climate.

In [ ]:
# Load ERA5 observational data for comparison
file_era5 = '../data/era5_global_t2m_monthly_lowres.nc'

try:
    ds_era5 = xr.open_dataset(file_era5)
    t2m_era5 = ds_era5.t2m  # Already in °C or K?
    
    # Check and convert units if needed
    if t2m_era5.max() > 100:  # Likely in Kelvin
        t2m_era5 = t2m_era5 - 273.15
        t2m_era5.attrs['units'] = '°C'
    
    print("✓ ERA5 observational data loaded")
    print(f"ERA5 period: {pd.to_datetime(t2m_era5.time[0].values).year} - {pd.to_datetime(t2m_era5.time[-1].values).year}")
    
except FileNotFoundError:
    print("Note: ERA5 data not found. Evaluation section will be skipped.")
    print("Download from: https://drive.google.com/file/d/15guRwxs56L9AJu32kSqz8A7zXxbtgXAx/view?usp=sharing")
    t2m_era5 = None

### Model Evaluation Metrics

We compute standard performance metrics:

**1. Bias:**
$$\text{Bias} = \frac{1}{n}\sum_{i=1}^{n}(\text{Model}_i - \text{Obs}_i)$$

**2. Root Mean Square Error (RMSE):**
$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(\text{Model}_i - \text{Obs}_i)^2}$$

**3. Correlation coefficient (r):**
$$r = \frac{\text{cov}(\text{Model}, \text{Obs})}{\sigma_{\text{Model}} \cdot \sigma_{\text{Obs}}}$$

**Interpretation:**
* **Bias:** Systematic over/underestimation (should be near zero)
* **RMSE:** Overall error magnitude (lower is better)
* **Correlation:** Pattern similarity (closer to 1 is better)

In [ ]:
if t2m_era5 is not None:
    # Compute yearly means for overlap period
    t2m_era5_yearly = t2m_era5.groupby('time.year').mean(dim='time')
    
    # Find common period (e.g., 1950-2014)
    years_overlap = np.intersect1d(
        tas_hist_global.year.values,
        t2m_era5_yearly.year.values
    )
    
    # Extract common period
    model_hist = tas_hist_global.sel(year=years_overlap)
    obs_hist = t2m_era5_yearly.mean(dim=('lat', 'lon')).sel(year=years_overlap)
    
    # Compute metrics
    bias = float((model_hist - obs_hist).mean())
    rmse = float(np.sqrt(((model_hist - obs_hist)**2).mean()))
    correlation = float(xr.corr(model_hist, obs_hist))
    
    # Compute trend comparison
    result_model = stats.linregress(years_overlap, model_hist.values)
    result_obs = stats.linregress(years_overlap, obs_hist.values)
    
    trend_model = result_model.slope * 10  # Per decade
    trend_obs = result_obs.slope * 10  # Per decade
    trend_bias = trend_model - trend_obs
    
    # Print evaluation report
    print("=" * 70)
    print("MODEL EVALUATION: MPI-ESM1-2-LR vs ERA5 Observations")
    print("=" * 70)
    print(f"Evaluation period: {years_overlap[0]} - {years_overlap[-1]}")
    print(f"Number of years: {len(years_overlap)}")
    print()
    print("BIAS AND ERROR:")
    print(f"  Mean bias: {bias:+.3f}°C")
    if abs(bias) < 0.5:
        print("    ✓ Acceptable bias (< 0.5°C)")
    else:
        print(f"    ⚠ Significant bias (> 0.5°C)")
    print(f"  RMSE: {rmse:.3f}°C")
    print(f"  Correlation: {correlation:.3f}")
    if correlation > 0.9:
        print("    ✓ Excellent correlation (r > 0.9)")
    elif correlation > 0.7:
        print("    ✓ Good correlation (r > 0.7)")
    else:
        print("    ⚠ Poor correlation (r < 0.7)")
    print()
    print("TREND COMPARISON:")
    print(f"  Model trend: {trend_model:+.4f}°C/decade (p = {result_model.pvalue:.2e})")
    print(f"  Observed trend: {trend_obs:+.4f}°C/decade (p = {result_obs.pvalue:.2e})")
    print(f"  Trend bias: {trend_bias:+.4f}°C/decade ({abs(trend_bias/trend_obs)*100:.1f}% error)")
    if abs(trend_bias/trend_obs) < 0.2:
        print("    ✓ Model captures observed warming trend well (< 20% error)")
    else:
        print("    ⚠ Significant trend bias (> 20% error)")
    print("=" * 70)
    
    # Visualization
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # Plot 1: Time series comparison
    ax1.plot(years_overlap, model_hist.values, 'b-', linewidth=2, label='MPI-ESM1-2-LR', alpha=0.7)
    ax1.plot(years_overlap, obs_hist.values, 'r-', linewidth=2, label='ERA5 Observations')
    ax1.set_xlabel('Year', fontsize=11)
    ax1.set_ylabel('Global Mean Temperature (°C)', fontsize=11)
    ax1.set_title('Model vs Observations', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Add text box with metrics
    textstr = f'Bias: {bias:+.3f}°C\nRMSE: {rmse:.3f}°C\nCorr: {correlation:.3f}'
    ax1.text(0.05, 0.95, textstr, transform=ax1.transAxes, fontsize=10,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Plot 2: Scatter plot
    ax2.scatter(obs_hist.values, model_hist.values, alpha=0.6, s=50)
    
    # Add 1:1 line
    min_val = min(obs_hist.min().values, model_hist.min().values)
    max_val = max(obs_hist.max().values, model_hist.max().values)
    ax2.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2, label='1:1 line')
    
    # Add regression line
    z = np.polyfit(obs_hist.values, model_hist.values, 1)
    p = np.poly1d(z)
    ax2.plot(obs_hist.values, p(obs_hist.values), 'r-', linewidth=2, 
             label=f'Fit: y={z[0]:.2f}x+{z[1]:.2f}')
    
    ax2.set_xlabel('ERA5 Observations (°C)', fontsize=11)
    ax2.set_ylabel('MPI-ESM1-2-LR Model (°C)', fontsize=11)
    ax2.set_title('Model vs Observations Scatter', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal', adjustable='box')
    
    plt.tight_layout()
    plt.savefig('model_evaluation.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("Skipping evaluation - ERA5 data not available")

---
## Computing Anomalies with Uncertainty

To avoid model bias issues, we compute **anomalies relative to a reference period** (1961-1990).

In [ ]:
# Compute yearly means
tas_hist_yearly = tas_hist.groupby('time.year').mean(dim='time')
tas_ssp1_yearly = tas_ssp1.groupby('time.year').mean(dim='time')
tas_ssp2_yearly = tas_ssp2.groupby('time.year').mean(dim='time')
tas_ssp3_yearly = tas_ssp3.groupby('time.year').mean(dim='time')
tas_ssp5_yearly = tas_ssp5.groupby('time.year').mean(dim='time')

# Compute climatology from historical period (1961-1990)
tas_clim_hist = tas_hist_yearly.sel(year=slice(1961, 1990)).mean(dim='year')
tas_std_hist = tas_hist_yearly.sel(year=slice(1961, 1990)).std(dim='year')

print(f"Reference period: 1961-1990")
print(f"Baseline global mean: {float(tas_clim_hist.mean()):.2f}°C")
print(f"Natural variability (std): {float(tas_std_hist.mean()):.2f}°C")

# Compute anomalies
tas_hist_anom = tas_hist_yearly - tas_clim_hist
tas_ssp1_anom = tas_ssp1_yearly - tas_clim_hist
tas_ssp2_anom = tas_ssp2_yearly - tas_clim_hist
tas_ssp3_anom = tas_ssp3_yearly - tas_clim_hist
tas_ssp5_anom = tas_ssp5_yearly - tas_clim_hist

print("\n✓ Anomalies computed relative to 1961-1990 baseline")

### Comparing model projections with observations

In [ ]:
if t2m_era5 is not None:
    # Compute ERA5 anomalies
    t2m_era5_yearly = t2m_era5.groupby('time.year').mean(dim='time')
    t2m_era5_clim = t2m_era5_yearly.sel(year=slice(1961, 1990)).mean(dim='year')
    t2m_era5_anom = t2m_era5_yearly - t2m_era5_clim
    
    # Global means
    obs_anom_global = t2m_era5_anom.mean(dim=('lat', 'lon'))
    
    print("✓ ERA5 anomalies computed")
else:
    obs_anom_global = None

In [ ]:
# Enhanced visualization with observations
plt.figure(figsize=(16, 8))

# Compute global mean anomalies
hist_anom_global = tas_hist_anom.mean(dim=('lat', 'lon'))
ssp1_anom_global = tas_ssp1_anom.mean(dim=('lat', 'lon'))
ssp2_anom_global = tas_ssp2_anom.mean(dim=('lat', 'lon'))
ssp3_anom_global = tas_ssp3_anom.mean(dim=('lat', 'lon'))
ssp5_anom_global = tas_ssp5_anom.mean(dim=('lat', 'lon'))

# Plot model projections
plt.plot(hist_anom_global.year, hist_anom_global.values, 
         color='black', linewidth=2.5, label='Historical (MPI-ESM1-2-LR)', zorder=5)
plt.plot(ssp1_anom_global.year, ssp1_anom_global.values, 
         color='darkgreen', linewidth=2.5, label='SSP1-2.6', alpha=0.9)
plt.plot(ssp2_anom_global.year, ssp2_anom_global.values, 
         color='blue', linewidth=2.5, label='SSP2-4.5', alpha=0.9)
plt.plot(ssp3_anom_global.year, ssp3_anom_global.values, 
         color='orange', linewidth=2.5, label='SSP3-7.0', alpha=0.9)
plt.plot(ssp5_anom_global.year, ssp5_anom_global.values, 
         color='red', linewidth=2.5, label='SSP5-8.5', alpha=0.9)

# Add ERA5 observations if available
if obs_anom_global is not None:
    plt.plot(obs_anom_global.year, obs_anom_global.values, 
             color='purple', linewidth=3, label='ERA5 Observations', 
             marker='o', markersize=3, zorder=6)

# Add reference lines
plt.axhline(y=0, color='gray', linestyle='--', linewidth=1.5, alpha=0.7, label='1961-1990 baseline')
plt.axhline(y=1.5, color='brown', linestyle=':', linewidth=2, alpha=0.7, label='1.5°C Paris target')
plt.axhline(y=2.0, color='darkred', linestyle=':', linewidth=2, alpha=0.7, label='2.0°C Paris target')
plt.axvline(x=2015, color='gray', linestyle='--', linewidth=1, alpha=0.5)

# Annotations
plt.text(2015, -0.5, 'Projection\nstart', ha='center', fontsize=9, color='gray')

plt.xlabel('Year', fontsize=12)
plt.ylabel('Temperature Anomaly (°C relative to 1961-1990)', fontsize=12)
plt.title('Global Temperature Anomalies: CMIP6 Projections vs Observations\n' +
          'MPI-ESM1-2-LR Model with Paris Agreement Targets',
          fontsize=13, fontweight='bold')
plt.legend(loc='upper left', fontsize=10, ncol=2)
plt.grid(True, alpha=0.3)
plt.xlim(1950, 2100)
plt.tight_layout()
plt.savefig('temperature_anomalies_with_targets.png', dpi=300, bbox_inches='tight')
plt.show()

### End-of-century warming projections

**IPCC AR6 assesses warming relative to 1850-1900.** Here we show relative to 1961-1990 and 1995-2014.

In [ ]:
# Compute end-of-century means (2081-2100)
period_end = slice(2081, 2100)
period_recent = slice(1995, 2014)

# End of century warming
ssp1_end = float(ssp1_anom_global.sel(year=period_end).mean())
ssp2_end = float(ssp2_anom_global.sel(year=period_end).mean())
ssp3_end = float(ssp3_anom_global.sel(year=period_end).mean())
ssp5_end = float(ssp5_anom_global.sel(year=period_end).mean())

# Recent period warming
if obs_anom_global is not None:
    recent_obs = float(obs_anom_global.sel(year=period_recent).mean())
else:
    recent_obs = None

print("=" * 70)
print("END-OF-CENTURY WARMING PROJECTIONS (2081-2100)")
print("Relative to 1961-1990 baseline")
print("=" * 70)
print(f"\nMPI-ESM1-2-LR Model:")
print(f"  SSP1-2.6: {ssp1_end:+.2f}°C")
print(f"  SSP2-4.5: {ssp2_end:+.2f}°C")
print(f"  SSP3-7.0: {ssp3_end:+.2f}°C")
print(f"  SSP5-8.5: {ssp5_end:+.2f}°C")

if recent_obs is not None:
    print(f"\nObserved warming (1995-2014 vs 1961-1990): {recent_obs:+.2f}°C")

print("\n" + "=" * 70)
print("IPCC AR6 WG1 BEST ESTIMATES (2081-2100 vs 1850-1900):")
print("=" * 70)
print("Multi-model mean projections:")
print("  SSP1-2.6: +1.8°C (likely range: 1.3-2.4°C)")
print("  SSP2-4.5: +2.7°C (likely range: 2.1-3.5°C)")
print("  SSP3-7.0: +3.6°C (likely range: 2.8-4.6°C)")
print("  SSP5-8.5: +4.4°C (likely range: 3.3-5.7°C)")
print("\nNote: Single model projections differ from multi-model means.")
print("      For policy, use IPCC multi-model assessments.")
print("=" * 70)

---
## Ensemble Statistics and Uncertainty Quantification

### Why use multiple models?

**Single model limitations:**
* Each model has unique biases
* Structural uncertainties
* Different parameterizations

**Multi-model ensemble advantages:**
* **Ensemble mean:** More reliable than any single model
* **Ensemble spread:** Quantifies model uncertainty
* **Robustness:** Agreement across models increases confidence

**Types of uncertainty:**
1. **Scenario uncertainty:** Which emissions pathway will we follow?
2. **Model uncertainty:** How well do models represent physics?
3. **Internal variability:** Natural climate fluctuations

**IPCC approach:**
* Uses 30-50 models per scenario
* Reports multi-model mean and spread
* Assesses agreement and robustness
* Uses likelihood language (likely = 66-100%, very likely = 90-100%)

### Simulating multi-model uncertainty

For demonstration, we'll simulate what a multi-model ensemble looks like by adding realistic uncertainty ranges to our single model.

In [ ]:
# Simulate inter-model spread based on IPCC AR6 ranges
# These represent typical ±1σ model spread for each scenario

np.random.seed(42)  # For reproducibility

# Define uncertainty ranges (approximate from IPCC AR6)
# These grow over time as uncertainty increases
years_proj = ssp1_anom_global.year.values
time_factor = (years_proj - 2015) / (2100 - 2015)  # 0 in 2015, 1 in 2100

# Scenario-dependent uncertainty (larger for high emissions)
uncertainty_ssp1 = 0.2 + 0.4 * time_factor  # Low scenario: smaller uncertainty
uncertainty_ssp2 = 0.3 + 0.6 * time_factor  # Medium
uncertainty_ssp3 = 0.4 + 0.8 * time_factor  # High
uncertainty_ssp5 = 0.5 + 1.0 * time_factor  # Highest: largest uncertainty

# Create uncertainty bounds (±1σ)
ssp1_lower = ssp1_anom_global - uncertainty_ssp1
ssp1_upper = ssp1_anom_global + uncertainty_ssp1

ssp2_lower = ssp2_anom_global - uncertainty_ssp2
ssp2_upper = ssp2_anom_global + uncertainty_ssp2

ssp3_lower = ssp3_anom_global - uncertainty_ssp3
ssp3_upper = ssp3_anom_global + uncertainty_ssp3

ssp5_lower = ssp5_anom_global - uncertainty_ssp5
ssp5_upper = ssp5_anom_global + uncertainty_ssp5

print("✓ Simulated multi-model uncertainty ranges")
print(f"\nEnd-of-century uncertainty (±1σ):")
print(f"  SSP1-2.6: ±{uncertainty_ssp1[-1]:.2f}°C")
print(f"  SSP2-4.5: ±{uncertainty_ssp2[-1]:.2f}°C")
print(f"  SSP3-7.0: ±{uncertainty_ssp3[-1]:.2f}°C")
print(f"  SSP5-8.5: ±{uncertainty_ssp5[-1]:.2f}°C")

In [ ]:
# Create comprehensive uncertainty plot
fig, ax = plt.subplots(figsize=(16, 9))

# Plot historical
ax.plot(hist_anom_global.year, hist_anom_global.values, 
        color='black', linewidth=2.5, label='Historical', zorder=5)

# Plot scenarios with uncertainty bands
# SSP1-2.6
ax.plot(years_proj, ssp1_anom_global.values, color='darkgreen', linewidth=2.5, 
        label='SSP1-2.6', zorder=4)
ax.fill_between(years_proj, ssp1_lower.values, ssp1_upper.values, 
                alpha=0.2, color='darkgreen', label='SSP1-2.6 uncertainty (±1σ)')

# SSP2-4.5
ax.plot(years_proj, ssp2_anom_global.values, color='blue', linewidth=2.5, 
        label='SSP2-4.5', zorder=4)
ax.fill_between(years_proj, ssp2_lower.values, ssp2_upper.values, 
                alpha=0.2, color='blue', label='SSP2-4.5 uncertainty (±1σ)')

# SSP3-7.0
ax.plot(years_proj, ssp3_anom_global.values, color='orange', linewidth=2.5, 
        label='SSP3-7.0', zorder=4)
ax.fill_between(years_proj, ssp3_lower.values, ssp3_upper.values, 
                alpha=0.2, color='orange', label='SSP3-7.0 uncertainty (±1σ)')

# SSP5-8.5
ax.plot(years_proj, ssp5_anom_global.values, color='red', linewidth=2.5, 
        label='SSP5-8.5', zorder=4)
ax.fill_between(years_proj, ssp5_lower.values, ssp5_upper.values, 
                alpha=0.2, color='red', label='SSP5-8.5 uncertainty (±1σ)')

# Add ERA5 if available
if obs_anom_global is not None:
    ax.plot(obs_anom_global.year, obs_anom_global.values, 
            color='purple', linewidth=3, label='ERA5 Observations', 
            marker='o', markersize=3, zorder=6)

# Reference lines
ax.axhline(y=0, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
ax.axhline(y=1.5, color='brown', linestyle=':', linewidth=2, alpha=0.7, label='1.5°C target')
ax.axhline(y=2.0, color='darkred', linestyle=':', linewidth=2, alpha=0.7, label='2.0°C target')
ax.axvline(x=2015, color='gray', linestyle='--', linewidth=1, alpha=0.3)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Temperature Anomaly (°C relative to 1961-1990)', fontsize=12)
ax.set_title('CMIP6 Climate Projections with Uncertainty\n' +
             'Shaded regions show approximate inter-model spread (simulated)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, ncol=2)
ax.grid(True, alpha=0.3)
ax.set_xlim(1950, 2100)
ax.set_ylim(-1, 6)

plt.tight_layout()
plt.savefig('cmip6_projections_with_uncertainty.png', dpi=300, bbox_inches='tight')
plt.show()

### Key findings and interpretation

**From this analysis:**

1. **Historical warming:**
   * Observed warming of ~1°C since 1961-1990
   * Model captures observed trend reasonably well
   * Some bias but correlation is good

2. **Future warming:**
   * All scenarios show continued warming
   * Scenario choice matters most after ~2040
   * 1.5°C will be crossed in all scenarios by ~2030-2035
   * 2°C crossed in SSP3 and SSP5 by mid-century

3. **Uncertainty:**
   * Grows with time (less certain about distant future)
   * Larger for high-emission scenarios
   * Model uncertainty is substantial
   * Near-term warming is more certain

4. **Policy implications:**
   * SSP1-2.6 needed to limit warming near 1.5-2°C
   * SSP2-4.5 leads to ~2.5-3°C warming
   * SSP5-8.5 could exceed 4-5°C
   * Every fraction of degree matters

**IPCC AR6 Assessment (2021):**
> "It is unequivocal that human influence has warmed the atmosphere, ocean and land. Widespread and rapid changes in the atmosphere, ocean, cryosphere and biosphere have occurred."

> "Global surface temperature will continue to increase until at least mid-century under all emissions scenarios considered. Global warming of 1.5°C and 2°C will be exceeded during the 21st century unless deep reductions in CO₂ and other greenhouse gas emissions occur in the coming decades."

---
## Summary and Key Takeaways

### What we learned:

**1. Global Climate Models:**
* Complex simulations of Earth's climate system
* CMIP6 provides standardized experiments
* Essential for understanding future climate

**2. SSP Scenarios:**
* Range from 1.9 to 8.5 W/m² forcing
* Include socioeconomic narratives
* SSP1-2.6 consistent with Paris Agreement
* SSP5-8.5 represents worst-case high emissions

**3. Model Evaluation:**
* Always evaluate against observations
* Compute bias, RMSE, correlation
* Check trend reproduction
* Use anomalies to avoid absolute bias

**4. Uncertainty:**
* Multiple sources (scenario, model, internal)
* Grows with projection length
* Multi-model ensembles essential
* Communicate uncertainty clearly

**5. IPCC AR6 Context:**
* Human influence is unequivocal
* 1.5°C will be reached in early 2030s
* Deep emissions cuts needed for Paris targets
* Every increment of warming matters

### Best Practices:

✅ **DO:**
* Use multi-model ensembles when possible
* Compute anomalies, not absolute values
* Evaluate model performance first
* Quantify and communicate uncertainty
* Reference IPCC assessments
* Consider multiple scenarios

❌ **DON'T:**
* Trust single model projections blindly
* Ignore model biases
* Use raw model output for local impacts
* Cherry-pick scenarios
* Neglect uncertainty

### Resources:

**Data:**
* CDS CMIP6: https://cds.climate.copernicus.eu/
* ESGF: https://esgf-node.llnl.gov/
* IPCC Interactive Atlas: https://interactive-atlas.ipcc.ch/

**Documentation:**
* IPCC AR6 WG1: https://www.ipcc.ch/report/ar6/wg1/
* CMIP6 overview: Eyring et al. (2016), GMD
* SSP scenarios: O'Neill et al. (2016), GMD

### Next Steps:
* **Notebook 6:** Climate attribution and extreme event analysis
* Explore regional climate projections
* Analyze climate extremes
* Learn about downscaling techniques